<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/12_spark_sql_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

spark = SparkSession.builder \
    .appName("SparkSQL") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

data = [("Raj",   "India", "Engineering", 50000, 25),
        ("Priya", "USA",   "Marketing",   70000, 28),
        ("Amit",  "India", "Engineering", 60000, 25),
        ("Sara",  "UK",    "HR",          80000, 32),
        ("Karan", "India", "Marketing",   45000, 27),
        ("John",  "USA",   "Engineering", 90000, 35),
        ("Neha",  "India", "HR",          55000, 29)]

columns = ["name", "country", "department", "salary", "age"]
df = spark.createDataFrame(data, columns)

# Register as temp view
df.createOrReplaceTempView("employees")

In [3]:
#DF API way

df.filter(f.col("salary")>50000).groupBy("department").agg(f.avg("salary")).show()

+-----------+-----------+
| department|avg(salary)|
+-----------+-----------+
|  Marketing|    70000.0|
|Engineering|    75000.0|
|         HR|    67500.0|
+-----------+-----------+



In [5]:
#spark sql way
df.createOrReplaceTempView("employees")
spark.sql("""
select department, avg(salary)    #both spark sql and df api goes through same execution plan/catalyst optimizer - hence there erfomance is same - no diff!
from employees
where salary >50000
group by department
""").show()   # returns df

+-----------+-----------+
| department|avg(salary)|
+-----------+-----------+
|  Marketing|    70000.0|
|Engineering|    75000.0|
|         HR|    67500.0|
+-----------+-----------+



In [6]:
#basic SELECT

spark.sql("""
select * from employees
""").show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|  Raj|  India|Engineering| 50000| 25|
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
|Karan|  India|  Marketing| 45000| 27|
| John|    USA|Engineering| 90000| 35|
| Neha|  India|         HR| 55000| 29|
+-----+-------+-----------+------+---+



In [7]:
#with WHERE

spark.sql("""
select * from employees where salary>50000
""").show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
| John|    USA|Engineering| 90000| 35|
| Neha|  India|         HR| 55000| 29|
+-----+-------+-----------+------+---+



In [8]:
#with ORDER BY

spark.sql("""
select * from employees where country='India' order by salary desc
""").show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
| Amit|  India|Engineering| 60000| 25|
| Neha|  India|         HR| 55000| 29|
|  Raj|  India|Engineering| 50000| 25|
|Karan|  India|  Marketing| 45000| 27|
+-----+-------+-----------+------+---+



In [9]:
#chain DataFrame operations on top

result = spark.sql(""" select * from employees where salary>55000""")
result.filter(f.col("country")=="India").show()

+----+-------+-----------+------+---+
|name|country| department|salary|age|
+----+-------+-----------+------+---+
|Amit|  India|Engineering| 60000| 25|
+----+-------+-----------+------+---+



## **GROUP BY in Spark SQL**

In [10]:
spark.sql(""" select department, count(*) as total_employees, avg(salary) as avg_salary, max(salary) as max_salary, min(salary) as min_salary from employees group by department""").show()

+-----------+---------------+-----------------+----------+----------+
| department|total_employees|       avg_salary|max_salary|min_salary|
+-----------+---------------+-----------------+----------+----------+
|  Marketing|              2|          57500.0|     70000|     45000|
|Engineering|              3|66666.66666666667|     90000|     50000|
|         HR|              2|          67500.0|     80000|     55000|
+-----------+---------------+-----------------+----------+----------+



## **JOINs in Spark SQL**

In [11]:
# Register second DataFrame
dept_data = [(101, "Engineering", "Mumbai"),
             (102, "Marketing",   "Delhi"),
             (103, "HR",          "Bangalore")]

dept_df = spark.createDataFrame(dept_data, ["dept_id", "dept_name", "location"])
dept_df.createOrReplaceTempView("departments")

In [12]:
dept_df.show()
df.show()

+-------+-----------+---------+
|dept_id|  dept_name| location|
+-------+-----------+---------+
|    101|Engineering|   Mumbai|
|    102|  Marketing|    Delhi|
|    103|         HR|Bangalore|
+-------+-----------+---------+

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|  Raj|  India|Engineering| 50000| 25|
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
|Karan|  India|  Marketing| 45000| 27|
| John|    USA|Engineering| 90000| 35|
| Neha|  India|         HR| 55000| 29|
+-----+-------+-----------+------+---+



In [17]:
#inner join

spark.sql(""" select e.name,e.salary, d.dept_name from employees e join departments d on e.department=d.dept_name""").show()

+-----+------+-----------+
| name|salary|  dept_name|
+-----+------+-----------+
|Priya| 70000|  Marketing|
|  Raj| 50000|Engineering|
| Amit| 60000|Engineering|
| Sara| 80000|         HR|
|Karan| 45000|  Marketing|
| Neha| 55000|         HR|
| John| 90000|Engineering|
+-----+------+-----------+



## **Subqueries in Spark SQL**

In [19]:
#find employees earning more than avg of salary

spark.sql("""select name, salary from employees where salary> (select avg(salary) from employees)""").show()

+-----+------+
| name|salary|
+-----+------+
|Priya| 70000|
| Sara| 80000|
| John| 90000|
+-----+------+



## **Practice Task**

Using the employees temp view:

Find all employees from India with salary above 50000

Find average salary per country using GROUP BY

Find the highest paid employee in each department using subquery

Join employees with a departments table you create

In [21]:
spark.sql("""select name, salary, country from employees where country='India' and salary>50000""").show()

+----+------+-------+
|name|salary|country|
+----+------+-------+
|Amit| 60000|  India|
|Neha| 55000|  India|
+----+------+-------+



In [22]:
spark.sql("""select country, avg(salary) from employees group by country""").show()

+-------+-----------+
|country|avg(salary)|
+-------+-----------+
|    USA|    80000.0|
|  India|    52500.0|
|     UK|    80000.0|
+-------+-----------+



In [27]:
spark.sql("""
    SELECT e.department, e.name, e.salary
    FROM employees e
    JOIN (
        SELECT department, MAX(salary) AS max_salary_in_dept
        FROM employees
        GROUP BY department
    ) AS max_salaries
    ON e.department = max_salaries.department AND e.salary = max_salaries.max_salary_in_dept
""").show()

+-----------+-----+------+
| department| name|salary|
+-----------+-----+------+
|  Marketing|Priya| 70000|
|Engineering| John| 90000|
|         HR| Sara| 80000|
+-----------+-----+------+



In [31]:
spark.sql("""select e.name, e.salary, d.dept_name from employees e join departments d on e.department= d.dept_name""").show()

+-----+------+-----------+
| name|salary|  dept_name|
+-----+------+-----------+
| John| 90000|Engineering|
| Amit| 60000|Engineering|
|  Raj| 50000|Engineering|
|Karan| 45000|  Marketing|
|Priya| 70000|  Marketing|
| Neha| 55000|         HR|
| Sara| 80000|         HR|
+-----+------+-----------+

